[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nube1/h-potential-companions/blob/main/methods-series/notebooks/00_quickstart.ipynb)

# H-potential in fifteen minutes

Six papers, six headline results, one pass. Each result gets about ten lines of code and
a plain statement of what it means. The notebooks numbered 01 to 06 take one result each
and do the work properly; this one is the map.

## Status, before anything else

- **Every datum here is synthetic.** Nothing in this notebook was measured on a person.
- **Papers 1 to 3 are protocols.** They say what will be done, not what was found. No
  participant has been recruited and no Delphi panel has met.
- **Papers 4 to 6 are method illustrations.** Their inputs show a workflow. They are not
  estimates from any health system and not claims about any product.
- **No figure below is evidence about patients.** Each number is a property of a stated
  model or a stated decision rule, and nothing more.

## Setup

The cell below installs the package straight from GitHub when this runs in Colab, and installs the local checkout when it does not. It prints the resolved version so you know what you are running.

**Runtime.** Reduced run: under one minute. Full run: about four minutes, almost all of it papers 1 and 2.

In [ ]:
# ---------------------------------------------------------------- setup
# Repository holding this package. The repository is private: in Colab, sign
# in to GitHub or clone it yourself before running.
GITHUB_OWNER = "Nube1"
REPO = "h-potential-companions"
SUBDIR = "methods-series"

import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q "hpotential[full] @ git+https://github.com/{GITHUB_OWNER}/{REPO}.git#subdirectory={SUBDIR}"
else:
    # Running from a clone: install the checkout this notebook sits in.
    import importlib.util
    import pathlib
    import subprocess

    if importlib.util.find_spec("hpotential") is None:
        for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (parent / "src" / "hpotential").exists():
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-q", "-e", str(parent)], check=True
                )
                break
        else:
            raise RuntimeError(
                "hpotential is not importable and no local checkout was found. "
                "Run this in Colab with access to the repository, or clone it."
            )

import hpotential

print(f"hpotential {hpotential.__version__}")
print(f"running in Colab: {IN_COLAB}")

In [ ]:
# Importing hpotential selects the Agg backend, because the library writes figures to
# files. Put the plots back into the page, then borrow the series figure style.
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpotential.utils.plotting import GRID, INK, apply_style, series_color

apply_style()
plt.rcParams["figure.dpi"] = 110  # 200 is for print; this is for a screen

try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except Exception:  # Colab sometimes ships without a working widget stack
    HAVE_WIDGETS = False


def explore(fn, controls, defaults):
    """Draw the static case first, then attach sliders if ipywidgets works.

    The static call means the notebook still teaches something when it is read on
    GitHub, where no widget ever runs.
    """
    fn(**defaults)
    if HAVE_WIDGETS:
        display(widgets.interactive(fn, **controls))
    else:
        print("ipywidgets is unavailable here, so the static case above is all there is.")


print("plots inline, style applied, widgets:", HAVE_WIDGETS)

In [ ]:
# ---------------------------------------------------------------- run size
FULL_RUN = False   # True reproduces the published values. Full run here takes roughly four minutes.


def banner():
    """Print at the top of every results cell. It has to be impossible to miss."""
    if FULL_RUN:
        print("=" * 74)
        print("FULL RUN: published replicate counts. These are the published values.")
        print("=" * 74)
    else:
        print("=" * 74)
        print("REDUCED RUN: cut replicate counts for speed.")
        print("The numbers below are NOT the published values. Set FULL_RUN = True.")
        print("=" * 74)


banner()

## 1. Paper 1: the determinacy ceiling

A second-order factor model puts indicators under domains and domains under one general
construct $\zeta$. The factor score $\widehat\zeta$ correlates with the truth at

$$\rho_D = \operatorname{corr}(\zeta, \widehat\zeta),$$

and no matter how many indicators are written, that correlation stops at

$$\rho_D^{\infty} = \sqrt{\frac{c}{1+c}}, \qquad c = \gamma^{\top}\Psi^{-1}\gamma .$$

The ceiling is set by the second-order part of the model, not by the item writing.

In [ ]:
banner()

from hpotential import determinacy_ceiling
from hpotential.core.measurement import required_second_order_signal, second_order_determinacy

rho_inf, c = determinacy_ceiling()
print(f"second-order signal c   = {c:.6f}")
print(f"ceiling rho_D_inf       = {rho_inf:.6f}")
print(f"at 400 indicators/domain= {second_order_determinacy(400, 0.85):.6f}   (still below)")
print(f"c needed for rho_D=0.90 = {required_second_order_signal(0.90):.4f}   (the design has 3.36)")

**What it means.** The planned design cannot deliver a score good enough for a decision
about one person, and writing more items will not fix it.
**What it does not mean.** It is not a statement that the construct is unmeasurable. Raise
the second-order loadings, or cut the domain disturbances, and the ceiling rises.

## 2. Paper 2: attenuation and the price in events

Fit a hazard model on the score rather than on the construct and the coefficient shrinks,
$\beta_{\text{obs}} \approx \rho_D\,\beta_{\text{true}}$. Matching the power of an oracle
study then needs $1/\rho_D^2$ times as many events.

In [ ]:
banner()

from hpotential.analyses.paper2_prognostic import population_model
from hpotential.config import Paper2Config
from hpotential.core.prognostic import attenuated_beta, events_inflation, required_events

cfg = Paper2Config()
rho_planned = population_model(cfg).determinacy()
print(f"rho_D of the planned design  {rho_planned:.4f}")
print(f"beta_true {cfg.beta_true:+.3f}  ->  expected fitted beta {attenuated_beta(cfg.beta_true, rho_planned):+.4f}")
for label, rho in [("oracle", 1.0), ("at the ceiling", rho_inf), ("planned design", rho_planned)]:
    print(f"events for 90% power, {label:<15} {required_events(cfg.beta_true, 0.90, 0.05, rho):7.0f}"
          f"   inflation {events_inflation(rho):.4f}")

**What it means.** 947 events is a floor. No instrument built on this second-order
structure beats it, so the extra 30 per cent over the oracle study is a design cost to
budget for, not a defect to engineer away.
**What it does not mean.** It does not say the study is infeasible.

## 3. Paper 3: what the Delphi rule actually does

The rule keeps an item when the median is at least 7, the interquartile range is at most
$\delta$, and the content-validity index is at least 0.78. At $\delta = 1.5$ the
dispersion limit is what does the dropping, and it drops good items more often as the
panel grows.

In [ ]:
banner()

from hpotential.core.consensus import operating_characteristics

n_sim = 20000 if FULL_RUN else 4000
rng = np.random.default_rng(20260815)
rows = []
for n_panel in (12, 20, 30, 45, 60):
    oc = operating_characteristics(n_panel, 0.90, 1.2, rng, n_sim, iqr_threshold=1.5)
    rows.append({"panel": n_panel, "false drop": oc["p_drop"], "IQR was binding": oc["iqr_fails"]})
pd.DataFrame(rows).round(3)

**What it means.** A rule meant to protect the item set is throwing well-endorsed items
away, and the larger the panel the worse it gets. The dispersion limit is the cause.
**What it does not mean.** These are properties of the rule under a stated rating model.
No panel has met, so this is not a finding about any expert group.

## 4. Paper 4: the solvers do what they claim

A continuous-time state model needs solvers that are checked, not asserted. Three checks:
fourth-order accuracy for RK4, strong order one for Euler-Maruyama with additive noise and
one half with multiplicative noise, and a Fokker-Planck scheme that conserves mass without
being renormalised.

In [ ]:
banner()

from hpotential.core.dynamical import (
    FokkerPlanck, LinearSDE, rk4, stable_time_step, strong_error_study,
)
from hpotential.utils.numerical import convergence_order

sde = LinearSDE(1.2, 0.6, 0.45)
steps = (0.4, 0.2, 0.1, 0.05, 0.025, 0.0125)
exact = sde.exact_solution(6.0, 3.0)
err = [abs(rk4(lambda t, x: sde.drift(x), 3.0, 6.0, h) - exact) for h in steps]
print(f"RK4 observed order {convergence_order(np.array(steps), np.array(err)):.3f}")

paths = 4000 if FULL_RUN else 800
for mult in (False, True):
    s = strong_error_study(LinearSDE(1.2, 0.6, 0.45, multiplicative=mult), 3.0, 6.0,
                           [75, 150, 300, 600, 1200], paths, np.random.default_rng(7), 9600)
    print(f"Euler-Maruyama strong order, {'multiplicative' if mult else 'additive':<14} "
          f"{s['strong_order']:.3f}")

fp = FokkerPlanck(sde, -2.0, 8.0)
_, _, dx = fp.grid(200)
sol = fp.solve(200, int(np.ceil(25.0 / stable_time_step(sde, dx, -2.0, 8.0, 0.4))), 25.0, x0=3.0)
print(f"Fokker-Planck mass drift {sol.mass_drift:.3e}   with no renormalisation")

**What it means.** The orders come out where the theory says, and the density solver keeps
its mass to machine precision because the scheme is conservative, not because anything was
rescaled afterwards.
**What it does not mean.** A verified solver says nothing about whether the model is the
right model.

## 5. Paper 5: costing the access-versus-efficacy choice

The uncosted comparison says spend on access when $T > A$. Put price and feasibility back
in and the rule becomes

$$V=\frac{T}{A}\cdot\frac{\Delta_A}{\Delta_T}\cdot\frac{c_T}{c_A} > 1 .$$

Two of the three factors are the ones the uncosted rule throws away.

In [ ]:
banner()

from hpotential.core.access import AccessModel

draws = 200_000 if FULL_RUN else 40_000
rng = np.random.default_rng(20260815)
rows = []
for model in (AccessModel.high_resource(), AccessModel.low_resource()):
    psa = model.psa(rng, draws)
    rows.append({"setting": model.name, "median V": psa["median_V"],
                 "uncosted favours access": psa["P_uncosted_rule_favours_access"],
                 "costed favours access": psa["P_costed_rule_favours_access"],
                 "rules disagree": psa["P_disagreement"]})
pd.DataFrame(rows).round(3)

**What it means.** In the high-resource setting the two rules point different ways on about
half the draws. The uncosted rule is not a safe shortcut there. In the low-resource setting
they never disagree, so the shortcut costs nothing.
**What it does not mean.** The input ranges are illustrative. The disagreement rate is a
property of those ranges, not a measurement of any health system.

## 6. Paper 6: early assessment, and what it is worth learning

Incremental net monetary benefit at a threshold $\lambda$ is
$\text{INB} = \lambda\,\Delta\text{QALY} - \Delta\text{cost}$. Expected value of perfect
information is $E[\max(\text{INB},0)] - \max(E[\text{INB}],0)$: what you would pay to stop
guessing.

In [ ]:
banner()

from hpotential.core.hta import HTAModel

draws = 100_000 if FULL_RUN else 20_000
model = HTAModel()
rng = np.random.default_rng(20260815)
rows = []
for scenario in ("optimistic", "central", "limited"):
    s = model.summarise(model.run_scenario(rng, scenario, draws))
    rows.append({"scenario": scenario, "P(cost-effective, QALY)": s["P_cost_effective_QALY"],
                 "P(cost-effective, BMP)": s["P_cost_effective_BMP"],
                 "decisions disagree": s["P_decision_disagreement"],
                 "per-person EVPI": s["per_person_EVPI"],
                 "BMP / QALY gain": s["mean_incremental_bmp"] / s["mean_incremental_qaly"]})
pd.DataFrame(rows).round(3)

**What it means.** Swap the outcome from QALYs to biomedical potential and the incremental
gain is about 1.9 times larger in the central scenario, which moves the decision. Two
outcome measures are not interchangeable just because both go up when health improves.
**What it does not mean.** Every input is synthetic. This is a workflow, not an appraisal.

## The six results on one page

In [ ]:
banner()

fig, axes = plt.subplots(2, 3, figsize=(11.5, 6.0))

ax = axes[0, 0]
ms = np.arange(3, 61)
for i, lam in enumerate((0.55, 0.65, 0.75, 0.85)):
    ax.plot(ms, [second_order_determinacy(int(m), lam) for m in ms], lw=1.3,
            color=series_color(i), label=rf"$\lambda$={lam}")
ax.axhline(rho_inf, ls=":", lw=1.2, color=series_color(5))
ax.axhline(0.90, ls="--", lw=0.9, color=INK)
ax.set_xlabel("indicators per domain"); ax.set_ylabel(r"$\rho_D$")
ax.set_title("1. the ceiling holds", fontsize=9); ax.legend(fontsize=6.5, ncol=2)

ax = axes[0, 1]
rhos = np.linspace(0.6, 1.0, 200)
ax.plot(rhos, [required_events(cfg.beta_true, 0.90, 0.05, r) for r in rhos], lw=1.5,
        color=series_color(0))
for r, lab in [(1.0, "oracle"), (rho_inf, "ceiling"), (rho_planned, "planned")]:
    ax.plot([r], [required_events(cfg.beta_true, 0.90, 0.05, r)], "o", ms=5,
            color=series_color(5))
    ax.annotate(lab, (r, required_events(cfg.beta_true, 0.90, 0.05, r)), fontsize=7,
                xytext=(-6, 8), textcoords="offset points", ha="right")
ax.set_xlabel(r"$\rho_D$"); ax.set_ylabel("events for 90% power")
ax.set_title("2. the price in events", fontsize=9)

ax = axes[0, 2]
panels = [12, 20, 30, 45, 60]
drops = [operating_characteristics(n, 0.90, 1.2, np.random.default_rng(n), n_sim,
                                   iqr_threshold=1.5)["p_drop"] for n in panels]
ax.plot(panels, drops, "o-", lw=1.4, ms=4, color=series_color(1))
ax.set_xlabel("panel size"); ax.set_ylabel("false-drop rate"); ax.set_ylim(0, 1)
ax.set_title("3. the rule drops good items", fontsize=9)

ax = axes[1, 0]
s_add = strong_error_study(LinearSDE(1.2, 0.6, 0.45), 3.0, 6.0, [75, 150, 300, 600, 1200],
                           paths, np.random.default_rng(7), 9600)
s_mul = strong_error_study(LinearSDE(1.2, 0.6, 0.45, multiplicative=True), 3.0, 6.0,
                           [75, 150, 300, 600, 1200], paths, np.random.default_rng(7), 9600)
ax.loglog(s_add["steps"], s_add["strong_errors"], "o-", ms=4, color=series_color(0),
          label=f"additive, {s_add['strong_order']:.2f}")
ax.loglog(s_mul["steps"], s_mul["strong_errors"], "s-", ms=4, color=series_color(1),
          label=f"multiplicative, {s_mul['strong_order']:.2f}")
ax.set_xlabel("step $h$"); ax.set_ylabel("strong error")
ax.set_title("4. strong order, one against one half", fontsize=9); ax.legend(fontsize=7)

ax = axes[1, 1]
hi = AccessModel.high_resource().draw(np.random.default_rng(1), 20000)
v = (hi["T"] / hi["A"]) * (hi["dA"] / hi["dT"]) * (hi["cT"] / hi["cA"])
ax.hist(np.log10(v), bins=60, color=series_color(2), edgecolor="none")
ax.axvline(0.0, ls="--", lw=1.0, color=INK)
ax.set_xlabel(r"$\log_{10} V$"); ax.set_ylabel("draws")
ax.set_title("5. $V>1$ on about half the draws", fontsize=9)

ax = axes[1, 2]
res = HTAModel().run_scenario(np.random.default_rng(3), "central", draws)
ax.scatter(res.incremental_qaly[:4000], res.incremental_cost[:4000] / 1000, s=2, alpha=0.25,
           color=series_color(3))
lam = np.linspace(0, res.incremental_qaly.max(), 50)
ax.plot(lam, 30.0 * lam, ls="--", lw=1.0, color=INK)
ax.set_xlabel("incremental QALYs"); ax.set_ylabel("incremental cost (thousands)")
ax.set_title("6. the cost-effectiveness plane", fontsize=9)

fig.tight_layout()
plt.show()

## What each paper claims, and what it does not

| paper | the claim | the limit of the claim |
| --- | --- | --- |
| 1. Measurement | For this second-order structure $\rho_D$ cannot pass 0.8778, and 0.90 needs $c \ge 4.2632$ against a planned 3.3586. The model is locally identified, rank 27 of 27. | Nothing about whether the instrument will have this $c$, and nothing about whether the model fits. No item has been administered. |
| 2. Prognostic | Attenuation lands near $\rho_D = 0.8361$ in simulation, and 90 per cent power needs about 1044 events against 730 for an oracle study. 947 is the floor at the ceiling. | A first-order regression-dilution result under non-informative measurement error. Simulated survival data, no cohort. |
| 3. Consensus | With the dispersion limit at 1.5 the false-drop rate for a well-endorsed item runs from 0.38 at 12 raters to 0.81 at 60, and dispersion is what binds on about 99 per cent of drops. At 2.0 the criterion cannot bind at all. | A property of the rule under a stated rating model and a stated quantile convention. No panel has met. |
| 4. Dynamics | RK4 order 4.05, Euler-Maruyama strong order 1.03 additive and about 0.53 multiplicative, mass conserved below 1e-15 with no renormalisation, first-order grid convergence. | Verification of solvers, not validation of the model. The parameters are illustrative. |
| 5. Access | $V$ reproduces the direct calculation to 6e-16, and the costed and uncosted rules disagree on about 48.6 per cent of high-resource draws and none of the low-resource ones. | Independent uniform inputs over illustrative ranges. $V>1$ ranks two programmes; it does not say either is worth funding. |
| 6. Early assessment | P(cost-effective on QALYs) is 0.410 in the central scenario, and the incremental biomedical potential is about 1.88 times the incremental QALY gain. | Synthetic inputs throughout. This is a workflow for an early decision, not an appraisal of a therapy. |

---

## What this does not establish

Nothing here is evidence about any patient, any clinician panel, any health system or any
product. Papers 1 to 3 are protocols and their numbers describe planned designs and stated
decision rules. Papers 4 to 6 use synthetic inputs picked to show a workflow.

A reduced run does not reproduce the published values. If `FULL_RUN` was `False` above,
treat every simulated number as an illustration of shape and sign, not of magnitude.

Nothing here says the H-potential construct exists, that the item bank measures it, or that
any of these models fits data that has not been collected.